# 06 - (Optional, Colab GPU) End-to-end generation with a free open LLM
Closes the loop from *retrieval depth* to **answer quality and real token usage**. Runs a small free reader (default `Qwen/Qwen2.5-0.5B-Instruct`; swap for `google/flan-t5-large`, `Qwen2.5-1.5B-Instruct`, ...) on **fixed top-k vs RCAK vs Adaptive-k**, logging per-query Exact Match, token-F1, prompt tokens (real tokenizer counts) and latency.

> This notebook needs Hugging Face model downloads, which are **blocked in the offline sandbox where the other notebooks were executed**, so it was *not* executed there. Run it on Colab (Runtime > T4 GPU). Use `N_EVAL = 300-500` to stay within free-tier limits.

**Outputs:** `csv/generation_per_query.csv`, `csv/generation_summary.csv`, `figures/generation_tradeoff.png`

In [ ]:
# ---- Environment setup (works in Colab AND in a local venv / Jupyter) ----
import os, sys, pathlib, subprocess, zipfile
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu", "sentence-transformers", "python-docx", "psutil"], check=False)
    if not pathlib.Path("adaptive_rag").exists():
        from google.colab import files
        print("Upload rag_adaptive_project.zip (the project archive you downloaded)")
        up = files.upload(); z = [k for k in up if k.endswith(".zip")][0]
        zipfile.ZipFile(z).extractall("."); 
        root = [p for p in pathlib.Path(".").iterdir() if (p / "adaptive_rag").exists()][0]; os.chdir(root)
    try:
        from google.colab import drive; drive.mount("/content/drive")      # results persist on your Drive
        os.environ["RAG_RESULTS_DIR"] = "/content/drive/MyDrive/rag_adaptive_results"
    except Exception as e:
        os.environ["RAG_RESULTS_DIR"] = os.path.abspath("results")
    BACKEND = "st"          # real neural embeddings on Colab (GPU recommended: Runtime > Change runtime type > T4)
else:
    here = pathlib.Path.cwd()
    root = here.parent if here.name == "notebooks" else here
    sys.path.insert(0, str(root)); os.chdir(root)
    os.environ.setdefault("RAG_RESULTS_DIR", str(root / "results_notebooks"))
    BACKEND = "lsa"         # offline CPU baseline (TF-IDF+SVD). Set to "st" if sentence-transformers can download models.
sys.path.insert(0, os.getcwd())
CFG = dict(backend=BACKEND, st_model="sentence-transformers/all-MiniLM-L6-v2", n_queries=int(os.environ.get("RAG_NQ", 10000)), seed=0)
print("backend:", BACKEND, "| results ->", os.environ["RAG_RESULTS_DIR"])


In [ ]:
N_EVAL = 400; MODEL = "Qwen/Qwen2.5-0.5B-Instruct"; MAX_NEW = 24
import numpy as np, pandas as pd, time, re, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from adaptive_rag import experiment as E
from adaptive_rag.data import normalize_answer
from adaptive_rag.features import feature_groups
from adaptive_rag.strategies import fixed_k, adaptive_k_gap
from adaptive_rag.utils import results_dir
B = E.prepare_cached(CFG)
tok = AutoTokenizer.from_pretrained(MODEL); lm = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32).to("cuda" if torch.cuda.is_available() else "cpu").eval()

In [ ]:
serve = "hybrid_rrf"; R = B.lists[serve]; te = np.where(B.split == "test")[0][:N_EVAL]
cols = [c for g in feature_groups().values() for c in g]
models = E.train_models(B, cols, 0); rc, Xca, rca = models[("rcak", serve)]; rc.calibrate(Xca, rca, 0.10)
K = {"fixed k=3": fixed_k(len(te), 3), "fixed k=10": fixed_k(len(te), 10), "fixed k=20": fixed_k(len(te), 20),
     "Adaptive-k (buffer 5)": adaptive_k_gap(R.scores[te], buffer=5), "RCAK alpha=0.10": rc.predict_k(B.feat.loc[te, cols].to_numpy())}
def f1(pred, golds):
    best = 0.0
    for g in golds:
        p, t = normalize_answer(pred).split(), normalize_answer(g).split()
        c = sum(min(p.count(w), t.count(w)) for w in set(p))
        if c: pr, rc_ = c/len(p), c/len(t); best = max(best, 2*pr*rc_/(pr+rc_))
    return best
@torch.no_grad()
def answer(q, passages):
    ctx = "\n\n".join(f"[{i+1}] {p}" for i, p in enumerate(passages))
    msgs = [{"role":"system","content":"Answer the question using ONLY the passages. Reply with a short answer span."},
            {"role":"user","content":f"Passages:\n{ctx}\n\nQuestion: {q}\nAnswer:"}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(lm.device)
    t = time.perf_counter(); out = lm.generate(ids, max_new_tokens=MAX_NEW, do_sample=False)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip(), ids.shape[1], time.perf_counter() - t
rows = []
for name, ks in K.items():
    for j, i in enumerate(te):
        q = B.queries.question.iloc[i]; gold = B.queries.answers.iloc[i]
        passages = B.corpus.text.iloc[R.ids[i, :ks[j]]].tolist(); pred, ntok, dt = answer(q, passages)
        em = int(any(normalize_answer(pred) == normalize_answer(g) for g in gold))
        rows.append({"method": name, "qid": B.queries.qid.iloc[i], "k": int(ks[j]), "prompt_tokens": ntok, "latency_s": dt, "EM": em, "F1": f1(pred, gold),
                     "answer_in_context": int(B.ans[serve][i, :ks[j]].any()), "pred": pred})
gen = pd.DataFrame(rows); gen.to_csv(results_dir("csv")/"generation_per_query.csv", index=False)
summ = gen.groupby("method").agg(mean_k=("k","mean"), prompt_tokens=("prompt_tokens","mean"), latency_s=("latency_s","mean"), EM=("EM","mean"), F1=("F1","mean"), answer_in_context=("answer_in_context","mean")).reset_index()
summ.to_csv(results_dir("csv")/"generation_summary.csv", index=False); display(summ.round(3))

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6.4, 4))
for _, r in summ.iterrows(): ax.scatter(r.prompt_tokens, r.F1, s=60); ax.annotate(r.method, (r.prompt_tokens, r.F1), fontsize=8, xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("mean prompt tokens (real tokenizer)"); ax.set_ylabel("token F1"); ax.set_title("Answer quality vs prompt cost"); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(results_dir("figures")/"generation_tradeoff.png", dpi=300); plt.show()